# Notebook 4, run 3 — The model (final run)

Fine-tunes one model, with three encoders, end to end on Notebook 3's dataset to forecast which
production Java files are refactored before the next release. It reads all three views of a row
(code, history and discourse) and reports, for the training, validation and test rows: the **loss**,
the **confusion matrix**, **precision, recall and F1** at a threshold, and **ROC-AUC and average
precision (AP)** from the probabilities. Cell 10 measures how much each view contributes to each
forecast (Shapley values). Every stage is timed.

This is the third and final run of `notebook_4_model_CBERT`, with its own output folder. It keeps run
2's model, text budget (`CAP` 64) and settings, and changes what runs 1 and 2 showed needed changing:

| | runs 1 and 2 | run 3 | why |
|---|---|---|---|
| rows per training step | as many as fit in the window budget: about 5 in run 1, about 2 in run 2 | **16**, with gradient accumulation | two rows per step made run 2's updates noisy: its training loss never fell below 0.693 and its history view stopped contributing. 16 is in BERT's fine-tuning grid of batch sizes {16, 32} (Devlin et al., 2019) |
| the balanced draw each epoch | all refactored rows + as many non-refactored rows from the whole pool | the same, **per repository**: each repository's refactored rows + as many of its own non-refactored rows | the repositories' refactored shares differ (30% of commons-lang's training rows, 7% of junit's); run 1 learned them, forecasting 89% of commons-lang's training rows as refactored and 1.4% of junit's. Balanced per repository, every repository is 50/50 in training, so a file's repository says nothing about its label |
| the epoch that is kept | best validation F1 | best validation **AP** | AP is the chapter's primary measure and needs no threshold; F1 at its best threshold is a maximum over thresholds and jumps from epoch to epoch. The threshold is still the one that maximises validation F1 |
| loss | training steps only | training steps, the 20,000-row training sample and the validation rows after every epoch; training, validation and test rows at the end | to compare training and validation loss |
| confusion matrix | not reported | per part and repository (`results.csv`), and `confusion.png` | |
| each view's contribution | at the end only | also after every epoch, on the validation rows | shows during training whether a view stops contributing |
| run times | per stage | per stage and per epoch, with the peak GPU memory | |
| forecasts saved | test rows | training, validation and test rows | any later table can be built without training again |
| pilot | measures `CAP` | none: `CAP` 64 and 192 windows per micro-batch, as run 2's pilot measured; one memory check | |

Nothing else changes: the encoders, the windows, attention pooling, the learning rates, the optimiser,
the schedule, dropout, the epoch limits and the seed are run 2's.

**Input:** the dataset on Zenodo, version 2.1, [10.5281/zenodo.24117444](https://doi.org/10.5281/zenodo.24117444),
downloaded and checked in cell 2.

| view | encoder | one text | a text longer than the encoder's limit |
|---|---|---|---|
| code | `microsoft/codebert-base` | the file's skeleton, or one method | 510-token windows, no overlap; the window vectors are averaged |
| history | `microsoft/codebert-base` (its own copy) | one commit: its message + this file's diff | the same |
| discourse | `sentence-transformers/all-mpnet-base-v2` | one issue body, PR body, comment or review | 382-token windows, half overlap; averaged |

**The model.** Each window becomes one vector (CodeBERT: its first token; Sentence-BERT: the mean of
its tokens). A text's window vectors are averaged into one text vector. Attention pooling, a learned
weighted average, turns a row's text vectors into one vector per view; a row with no text in a view
gets a learned "empty" vector. The view vectors are joined and a linear layer gives the probability
that the file is refactored. All weights, the encoders' included, learn from the labels (end to end).

**One model**, trained on the five repositories' training rows together and scored per repository
and overall.

**Training.**
- An epoch is, for each repository, all its refactored training rows plus the same number of its
  non-refactored training rows drawn at random; a new draw each epoch.
- The epoch's rows are taken 16 at a time. Each group of 16 is one optimiser step: its rows go through
  the model in micro-batches of at most 192 windows, and their gradients are added up before the step.
  The step is the same as one batch of 16 rows; the micro-batches only keep it inside GPU memory.
- **Text budget:** each view of a row reads at most `CAP` = 64 windows: the skeleton first, then the
  methods largest first; the newest commits; the newest discourse texts.
- After each epoch the validation rows are forecast and the threshold that maximises F1 on them is
  found. Validation AP decides early stopping, and the best epoch is kept. The 20,000-row training
  sample is scored at the same threshold, to show overfitting.
- The **test** rows are used once, in cell 9, after training is done.

**Two training losses.** Training steps see a balanced draw (half refactored), while the training
sample, the validation and the test rows have their natural share of refactored rows (3–8%). The loss
of the steps is therefore not comparable with the validation loss. The training-sample loss is: it is
measured the same way as the validation loss, on the natural share. Compare those two.

| setting | value | why, and the source |
|---|---|---|
| learning rate, encoders | 2e-5 | in CodeBERT's fine-tuning grid {2e-5, 5e-5} (Feng et al., 2020) and BERT's {2e-5, 3e-5, 5e-5} (Devlin et al., 2019); the best of the three in run 1 |
| rows per step | 16 | BERT's fine-tuning grid of batch sizes {16, 32} (Devlin et al., 2019) |
| optimizer | AdamW, weight decay 0.01 | Loshchilov & Hutter, 2019; weight decay 0.01 as in BERT (Devlin et al., 2019) |
| learning rate, pooling and classifier | 1e-3 | these layers start from random weights and are tiny next to the encoders, so they get a larger rate; a design choice, not from a paper |
| schedule | linear warm-up over the first 10% of steps, then linear decay | BERT's fine-tuning schedule (Devlin et al., 2019, released code) |
| gradient clipping | 1.0 | BERT's released code |
| dropout | 0.2, on the joined view vectors | dropout: Srivastava et al., 2014; the rate is a design choice (the encoders keep their own 0.1) |
| precision | bfloat16 | Kalamkar et al., 2019 |
| memory | gradient checkpointing; gradient accumulation | Chen et al., 2016 |
| epochs | at most 10; stop after 3 without a better validation AP | early stopping (Prechelt, 1998); 10 and 3 are design choices |
| seed | 42 | a convention; fixes the epoch draws and the starting weights of the new layers |

**Writes** (to `notebook_4_model_cbert_3/outputs/`, on Google Drive on Colab and beside the notebook elsewhere):

| file | holds |
|---|---|
| `text_lengths.csv` | tokens per text, by kind of text |
| `results.csv` | per part (train, validation, test), per repository and overall: rows, refactored, the confusion matrix (TP, FP, FN, TN), loss, precision, recall, F1, ROC-AUC and AP |
| `run_times.csv` | minutes per epoch (training, validation scoring, training-sample scoring) with the peak GPU memory; the final scoring of each part; totals |
| `epochs.csv` | every epoch: the losses, the scores on the training sample and the validation rows, each view's contribution, the minutes |
| `forecasts.parquet` | one line per training, validation and test row: part, row_id, repo, release, path, label, probability, forecast |
| `view_shap_test.parquet` | one line per test row: each view's contribution to the forecast (log-odds) |
| `view_shap.csv` | each view's mean contribution, per repository and overall |
| `curves.png` | per epoch: loss, F1, AP and each view's contribution |
| `confusion.png` | the confusion matrices of the training, validation and test rows |
| `runs.jsonl` | the training run, epoch by epoch |
| `run_config.json` | the settings, the GPU and package versions |

**If the session stops** (a Colab disconnect or a pod restart), run all cells again: a finished run is skipped and an unfinished run resumes
from its last epoch.

| cell | what it does |
|---|---|
| 1 | setup and settings |
| 2 | download the dataset from Zenodo, unzip it and check it |
| 3 | pool the five repositories |
| 4 | cut every text into windows; text lengths |
| 5 | the model |
| 6 | the text budget and a memory check |
| 7 | one training run |
| 8 | the training run |
| 9 | the scores, the confusion matrices and the run times |
| 10 | each view's contribution (Shapley values) |
| 11 | curves and confusion matrices |

Needs a GPU with bfloat16 and the memory for 192 windows per micro-batch (run 2 used an RTX PRO
6000 with 96 GB), and about 12 GB free on disk (Google Drive on Colab). At run 2's speed an epoch takes about 100
minutes, so 10 epochs take about 17 hours.


### Cell 1 — Setup and settings
On Colab it mounts Google Drive. Elsewhere (RunPod, a local machine) it keeps everything in
`notebook_4_model_cbert_3` beside the notebook. The dataset, the checkpoints and the results live there, so a
disconnect loses nothing. Every setting of the experiment is in this cell.

In [ ]:
import os, json, math, time, gc, hashlib, itertools, functools, platform, shutil, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import sklearn
import transformers
from sklearn.metrics import (precision_recall_curve, precision_recall_fscore_support, roc_auc_score,
                             average_precision_score, log_loss)
from transformers import AutoTokenizer, AutoModel

transformers.logging.set_verbosity_error()          # silences "sequence longer than 512" on tokenizing

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/notebook_4_model_cbert_3")
except ImportError:
    WORK = Path.cwd() / "notebook_4_model_cbert_3"     # not on Colab: keep everything beside the notebook
DATA, CKPT, OUT = WORK / "data", WORK / "checkpoints", WORK / "outputs"
for d in (DATA, CKPT, OUT):
    d.mkdir(parents=True, exist_ok=True)
print(f"working folder {WORK}")

RECORD = "24117444"                                  # Zenodo record, version 2.1, DOI 10.5281/zenodo.24117444
REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]

ENCODER = {"code": "microsoft/codebert-base", "history": "microsoft/codebert-base",
           "discourse": "sentence-transformers/all-mpnet-base-v2"}
BODY = {"code": 510, "history": 510, "discourse": 382}       # tokens per window, without the two special tokens
STEP = {"code": 510, "history": 510, "discourse": 191}       # no overlap; half overlap
VIEWS = ["code", "history", "discourse"]                    # the model reads all three

LR = 2e-5                                                   # Feng et al., 2020; Devlin et al., 2019; best in run 1
LR_HEAD, WEIGHT_DECAY, DROPOUT, WARMUP, CLIP = 1e-3, 0.01, 0.2, 0.1, 1.0
EPOCHS, PATIENCE, SEED = 10, 3, 42
ROWS_PER_STEP = 16              # rows per optimiser step (Devlin et al., 2019: batch size 16 or 32)
SELECT = "ap"                   # the validation score that picks the epoch

CAP = 64                        # windows per view per row, as run 2's pilot measured
BATCH_WINDOWS = 192             # windows per micro-batch (3 x CAP: one row's largest possible share)
FIT_SCORED = 20_000             # training rows scored after each epoch, for the overfitting curves
EVAL_TOKENS = 65_536            # tokens per batch when encoding without gradients
KMAX = 64                       # no text is cut into more windows than this

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
torch.backends.cuda.matmul.allow_tf32 = True
print(f"work folder {WORK}\nGPU {GPU}   bfloat16 {'on' if BF16 else 'OFF'}")
if not BF16:
    print("This notebook needs a GPU with bfloat16 (Runtime > Change runtime type).")


### Cell 2 — The dataset, from Zenodo
Downloads the record’s five zips (one per repository) and unpacks their 30 files (six per repository)
into the working folder, once. Every data file is then checked against the SHA-256 that Notebook 3 wrote
into `manifest_<repo>.json`, so the model trains on exactly the files Notebook 3 produced. The cell
stops if any file differs.

In [ ]:
PARTS = [("rows", "parquet"), ("code_texts", "parquet"), ("history_texts", "parquet"),
         ("discourse_texts", "parquet"), ("split", "csv"), ("manifest", "json")]
NAMES = [f"{k}_{r}.{ext}" for r in REPOS for k, ext in PARTS]


def fetch(repo):
    """Download <repo>.zip once and unpack its six files into DATA (any folder inside the zip is ignored)."""
    want = {f"{k}_{repo}.{ext}" for k, ext in PARTS}
    if all((DATA / n).exists() for n in want):
        return
    url = f"https://zenodo.org/records/{RECORD}/files/{repo}.zip?download=1"
    zpath = DATA / f"{repo}.zip"
    try:
        urllib.request.urlretrieve(url, zpath)
    except urllib.error.HTTPError as e:
        raise SystemExit(f"{repo}.zip: {e} from {url}") from None
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            name = Path(m.filename).name
            if name in want:
                (DATA / name).write_bytes(z.read(m))
    zpath.unlink()
    missing = [n for n in want if not (DATA / n).exists()]
    if missing:
        raise SystemExit(f"{repo}.zip does not hold {missing}")
    print(f"{repo}: downloaded and unpacked")


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


import zipfile
for r in REPOS:
    fetch(r)
bad = []
for r in REPOS:
    for f in json.loads((DATA / f"manifest_{r}.json").read_text(encoding="utf-8"))["files"]:
        if sha256(DATA / f["file"]) != f["sha256"]:
            bad.append(f["file"])
if bad:
    raise SystemExit(f"these files differ from Notebook 3's manifest: {bad}")
size = sum((DATA / n).stat().st_size for n in NAMES)
print(f"{len(NAMES)} files, {size / 1e9:.2f} GB, all match Notebook 3's manifests")


### Cell 3 — The five repositories, pooled
Joins the five repositories' rows and the ids of their texts. Rows `left out` by Notebook 3's split are
dropped here and never used. Each row's text ids become positions in its view's text table. The cell
stops if the row counts differ from the dataset's description.

In [ ]:
KEY = {"code": "code_id", "history": "history_id", "discourse": "discourse_id"}

ROWS = pd.concat([pd.read_parquet(DATA / f"rows_{r}.parquet") for r in REPOS], ignore_index=True)
ROWS = ROWS[ROWS.split != "left out"].reset_index(drop=True)
TEXT = {v: pd.concat([pd.read_parquet(DATA / f"{v}_texts_{r}.parquet", columns=[KEY[v]]) for r in REPOS])
            .drop_duplicates(KEY[v]).reset_index(drop=True) for v in VIEWS}           # the text ids, in a fixed order
IDX = {}                                                     # view -> one int array per row
for v in VIEWS:
    pos = dict(zip(TEXT[v][KEY[v]], range(len(TEXT[v]))))
    IDX[v] = [np.array([pos[i] for i in ids], dtype=np.int64) for ids in ROWS[f"{v}_ids"]]
LABEL = ROWS.label.to_numpy().astype(np.float32)
FIT = np.flatnonzero(ROWS.split == "train")
VAL = np.flatnonzero(ROWS.split == "validation")
TEST = np.flatnonzero(ROWS.split == "test")

EXPECTED = {"train": (151_496, 12_346), "validation": (99_964, 3_510), "test": (200_438, 6_444)}   # version 2
got = {s: (int((ROWS.split == s).sum()), int(ROWS.label[ROWS.split == s].sum())) for s in EXPECTED}
print(pd.DataFrame(got, index=["rows", "refactored"]).T.to_string())
if got != EXPECTED:
    raise SystemExit(f"rows differ from the dataset's description: {EXPECTED}")
print("\n" + "   ".join(f"{v} texts {len(TEXT[v]):,}" for v in VIEWS))


### Cell 4 — Every text cut into windows, and how long the texts are
Each text is tokenised once with its view's tokenizer, and its windows are fixed: 510 tokens with no
overlap for CodeBERT, 382 tokens with half overlap for Sentence-BERT. A text keeps at most 64 windows.
The tokens are saved in the working folder and later sessions load them. Run 2's tokens are not
reused, because they belong to version 1 of the dataset (the cell checks the count). Then the code view is sorted into budget order (the skeleton first, then the
methods largest first), and each row's number of windows per view is counted.

The cell also reports how long each kind of text is, in tokens, before any cut, and writes the table
to `text_lengths.csv`.

In [ ]:
import re
def n_windows(n, v):
    return 1 if n <= BODY[v] else 1 + math.ceil((n - BODY[v]) / STEP[v])


TOKFILE = WORK / "tokens.npz"
TOK = {v: AutoTokenizer.from_pretrained(ENCODER[v]) for v in VIEWS}
if TOKFILE.exists():
    z = np.load(TOKFILE)
    TOKS = {v: (z[f"{v}_flat"], z[f"{v}_off"]) for v in VIEWS}
    FULL = {v: z[f"{v}_full"] for v in VIEWS}                # each text's length before any cut
    print(f"tokens loaded from {TOKFILE}")
else:
    TOKS, FULL = {}, {}
    for v in VIEWS:
        keep = BODY[v] + (KMAX - 1) * STEP[v]                 # tokens needed for 64 windows
        T = pd.concat([pd.read_parquet(DATA / f"{v}_texts_{r}.parquet", columns=[KEY[v], "text"])
                       for r in REPOS]).drop_duplicates(KEY[v])
        flat, lens, full, texts = [], [], [], T.text.tolist()
        del T
        for i in range(0, len(texts), 2000):
            ids = TOK[v](texts[i:i + 2000], add_special_tokens=False)["input_ids"]
            lens += [min(len(x), keep) for x in ids]
            full += [len(x) for x in ids]
            flat.append(np.fromiter(itertools.chain.from_iterable(x[:keep] for x in ids), dtype=np.int32))
            print(f"\r   {v}: {min(i + 2000, len(texts)):,} / {len(texts):,} texts", end="")
        TOKS[v] = (np.concatenate(flat), np.concatenate([[0], np.cumsum(lens)]).astype(np.int64))
        FULL[v] = np.array(full, dtype=np.int64)
        print()
    np.savez(TOKFILE, **{f"{v}_{k}": a for v in VIEWS for k, a in zip(("flat", "off"), TOKS[v])},
             **{f"{v}_full": FULL[v] for v in VIEWS})
    print(f"tokens saved to {TOKFILE}")
for v in VIEWS:
    if len(TOKS[v][1]) - 1 != len(TEXT[v]):
        raise SystemExit(f"{TOKFILE.name} does not match the dataset: delete it and run this cell again")

NTOK = {v: np.diff(TOKS[v][1]) for v in VIEWS}
NWIN = {v: np.array([min(n_windows(int(n), v), KMAX) for n in NTOK[v]]) for v in VIEWS}
IDX["code"] = [ids if len(ids) < 3 else np.concatenate([ids[:1], ids[1:][np.argsort(-NTOK["code"][ids[1:]],
               kind="stable")]]) for ids in IDX["code"]]     # skeleton first, then methods largest first
WTOT = {v: np.array([int(NWIN[v][ids].sum()) for ids in IDX[v]]) for v in VIEWS}


def window(v, t, j):
    """Window j of text t in view v, as token ids without the special tokens."""
    flat, off = TOKS[v]
    s = off[t] + j * STEP[v]
    return flat[s:min(s + BODY[v], off[t + 1])]


print("\nwindows per row before the budget (training rows)")
print(pd.DataFrame({v: pd.Series(WTOT[v][FIT]).describe(percentiles=[.5, .9, .99]).round(0) for v in VIEWS})
      .loc[["50%", "90%", "99%", "max"]].rename(index={"50%": "median", "90%": "90th percentile",
                                                       "99%": "99th percentile"}).to_string())


# ---- how long are the texts? (tokens before any cut)
LENFILE = OUT / "text_lengths.csv"
if LENFILE.exists():
    LENS = pd.read_csv(LENFILE)
else:
    parts = []
    for v in ("code", "discourse"):
        K = pd.concat([pd.read_parquet(DATA / f"{v}_texts_{r}.parquet", columns=[KEY[v], "kind"])
                       for r in REPOS]).drop_duplicates(KEY[v])
        parts.append(pd.DataFrame({"view": v, "kind": K.kind.to_numpy(), "tokens": FULL[v]}))
    H = pd.concat([pd.read_parquet(DATA / f"history_texts_{r}.parquet", columns=["history_id", "sha", "text"])
                   for r in REPOS]).drop_duplicates("history_id")
    msg = H.text.str.extract(r"<COMMIT_MESSAGE>(.*?)</COMMIT_MESSAGE>", flags=re.S)[0].fillna("")
    mtok = np.array([len(x) for x in TOK["history"](msg.tolist(), add_special_tokens=False)["input_ids"]])
    one = ~H.sha.duplicated().to_numpy()                      # each commit's message once
    parts.append(pd.DataFrame({"view": "history", "kind": "commit message", "tokens": mtok[one]}))
    parts.append(pd.DataFrame({"view": "history", "kind": "diff part (text minus message)",
                               "tokens": np.maximum(0, FULL["history"] - mtok)}))
    parts.append(pd.DataFrame({"view": "history", "kind": "whole text (message + diff)", "tokens": FULL["history"]}))
    del H, msg
    A = pd.concat(parts, ignore_index=True)
    A["over"] = A.tokens > A.view.map(BODY)
    LENS = A.groupby(["view", "kind"], sort=False).agg(
        texts=("tokens", "size"), median=("tokens", "median"), p90=("tokens", lambda x: x.quantile(.9)),
        p99=("tokens", lambda x: x.quantile(.99)), max=("tokens", "max"), longer_than_one_window=("over", "mean")
    ).reset_index()
    LENS[["median", "p90", "p99"]] = LENS[["median", "p90", "p99"]].round(0).astype(int)
    LENS["longer_than_one_window"] = (100 * LENS.longer_than_one_window).round(1)
    LENS.to_csv(LENFILE, index=False)
print("\ntokens per text, before any cut (one window = 510 tokens; 382 for discourse)")
print(LENS.rename(columns={"p90": "90th", "p99": "99th", "longer_than_one_window": "% longer than one window"})
      .to_string(index=False))


### Cell 5 — The model
`row_plan` applies the text budget: a view of a row reads its texts in order (code: skeleton, then
methods largest first; history and discourse: newest first) until `cap` windows are taken; the last
text may contribute only its first windows.

`Forecaster` holds one encoder per view, one attention pooling per view, and the classifier. During
training the windows of a micro-batch go through the encoders with gradients. For scoring, the
weights do not change, so `predict` encodes each distinct window once, however many rows share it,
and pools from those vectors.

`scores` gives, at a threshold, the confusion matrix, precision, recall and F1, and from the
probabilities the loss (binary cross-entropy), ROC-AUC and AP.

In [ ]:
def row_plan(r, v, cap):
    """[(text, windows taken)] for row r's view v under the budget."""
    out, left = [], cap
    for t in IDX[v][r]:
        if left == 0:
            break
        k = min(int(NWIN[v][t]), left)
        out.append((int(t), k))
        left -= k
    return out


def pad(v, wins):
    """Token-id windows -> (ids, attention mask) with the special tokens added."""
    tok = TOK[v]
    L = max([len(w) for w in wins], default=0) + 2
    ids = np.full((len(wins), L), tok.pad_token_id, dtype=np.int64)
    att = np.zeros((len(wins), L), dtype=np.int64)
    for i, w in enumerate(wins):
        ids[i, 0], ids[i, 1:len(w) + 1], ids[i, len(w) + 1] = tok.cls_token_id, w, tok.sep_token_id
        att[i, :len(w) + 2] = 1
    return torch.from_numpy(ids), torch.from_numpy(att)


def make_batch(rows, views, cap):
    batch = {"n": len(rows), "y": torch.from_numpy(LABEL[rows])}
    for v in views:
        wins, win_text, text_row = [], [], []
        for b, r in enumerate(rows):
            for t, k in row_plan(r, v, cap):
                text_row.append(b)
                for j in range(k):
                    wins.append(window(v, t, j))
                    win_text.append(len(text_row) - 1)
        batch[v] = (*pad(v, wins), torch.tensor(win_text, dtype=torch.long),
                    torch.tensor(text_row, dtype=torch.long))
    return batch


def checkpointed(forward, layer, *args, **kwargs):
    """Recompute this layer in the backward pass instead of storing its activations."""
    if layer.training and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(forward, *args, use_reentrant=False, **kwargs)
    return forward(*args, **kwargs)


class Encoder(nn.Module):
    def __init__(self, name, first_token):
        super().__init__()
        self.m = AutoModel.from_pretrained(name)
        if self.m.supports_gradient_checkpointing:
            self.m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        else:                                   # MPNet has no built-in switch: checkpoint each layer by hand
            for layer in self.m.encoder.layer:
                layer.forward = functools.partial(checkpointed, layer.forward, layer)
        self.first_token = first_token          # CodeBERT: the first token; Sentence-BERT: the mean

    def forward(self, ids, att):
        h = self.m(input_ids=ids, attention_mask=att).last_hidden_state
        if self.first_token:
            return h[:, 0]
        m = att.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)


class Pool(nn.Module):
    """Window vectors -> text vectors (mean) -> one vector per row (attention pooling)."""

    def __init__(self, hid):
        super().__init__()
        self.score = nn.Linear(hid, 1)
        self.empty = nn.Parameter(torch.zeros(hid))

    def forward(self, h, win_text, text_row, n):
        h = h.float()
        n_text = text_row.numel()
        t = torch.zeros(n_text, h.size(1), device=h.device).index_add_(0, win_text, h)
        t = t / torch.zeros(n_text, device=h.device).index_add_(
            0, win_text, torch.ones(len(win_text), device=h.device)).clamp(min=1).unsqueeze(1)
        s = self.score(t).squeeze(1).float()
        top = torch.full((n,), -1e30, device=h.device).scatter_reduce(0, text_row, s, "amax")
        e = torch.exp(s - top[text_row])
        a = e / torch.zeros(n, device=h.device).index_add_(0, text_row, e)[text_row]
        v = torch.zeros(n, h.size(1), device=h.device).index_add_(0, text_row, t * a.unsqueeze(1))
        has = torch.zeros(n, dtype=torch.bool, device=h.device)
        has[text_row] = True
        return torch.where(has.unsqueeze(1), v, self.empty.float().expand(n, -1))


class Forecaster(nn.Module):
    def __init__(self, views):
        super().__init__()
        self.views = views
        self.enc = nn.ModuleDict({v: Encoder(ENCODER[v], first_token=(v != "discourse")) for v in views})
        hid = self.enc[views[0]].m.config.hidden_size
        self.pool = nn.ModuleDict({v: Pool(hid) for v in views})
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(hid * len(views), 1)

    def joined(self, parts, n):
        """parts: view -> (window vectors, window -> text, text -> row) -> the three view vectors, joined."""
        return torch.cat([self.pool[v](*parts[v], n) for v in self.views], 1)

    def classify(self, parts, n):
        return self.out(self.drop(self.joined(parts, n))).squeeze(1)

    def forward(self, batch):
        parts = {}
        for v in self.views:
            ids, att, win_text, text_row = (x.to(DEVICE) for x in batch[v])
            hid = self.out.in_features // len(self.views)
            h = self.enc[v](ids, att) if len(ids) else torch.zeros(0, hid, device=DEVICE)
            parts[v] = (h, win_text, text_row)
        return self.classify(parts, batch["n"])


@torch.no_grad()
def encode_windows(enc, v, keys):
    """Distinct windows (key = text * KMAX + window) -> vectors, shortest windows batched together."""
    wins = [window(v, int(k // KMAX), int(k % KMAX)) for k in keys]
    lens = np.array([len(w) for w in wins])
    order = np.argsort(lens, kind="stable")
    hid = enc.m.config.hidden_size
    W = torch.empty(len(keys), hid, dtype=torch.bfloat16 if BF16 else torch.float32, device=DEVICE)
    i = 0
    while i < len(order):
        e = i + 1
        while e < len(order) and (e + 1 - i) * (lens[order[e]] + 2) <= EVAL_TOKENS:
            e += 1
        ids, att = pad(v, [wins[k] for k in order[i:e]])
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            W[torch.from_numpy(order[i:e]).to(DEVICE)] = enc(ids.to(DEVICE), att.to(DEVICE)).to(W.dtype)
        i = e
    return W


@torch.no_grad()
def predict(model, rows, cap, chunk=8192, keep_views=False):
    """Probability of refactoring for each row, encoding each distinct window once.
    keep_views=True also returns each row's joined view vectors (for cell 10)."""
    model.eval()
    vec = {}
    for v in model.views:
        keys = np.unique(np.fromiter((t * KMAX + j for r in rows for t, k in row_plan(r, v, cap)
                                      for j in range(k)), dtype=np.int64))
        vec[v] = (keys, encode_windows(model.enc[v], v, keys))
    probs, joined = [], []
    for s in range(0, len(rows), chunk):
        part, parts = rows[s:s + chunk], {}
        for v in model.views:
            keys, W = vec[v]
            wk, win_text, text_row = [], [], []
            for b, r in enumerate(part):
                for t, k in row_plan(r, v, cap):
                    text_row.append(b)
                    for j in range(k):
                        wk.append(t * KMAX + j)
                        win_text.append(len(text_row) - 1)
            at = torch.from_numpy(np.searchsorted(keys, np.array(wk, dtype=np.int64))).to(DEVICE)
            parts[v] = (W[at], torch.tensor(win_text, dtype=torch.long, device=DEVICE),
                        torch.tensor(text_row, dtype=torch.long, device=DEVICE))
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            x = model.joined(parts, len(part))
            probs.append(torch.sigmoid(model.out(x).squeeze(1).float()).cpu().numpy())
        if keep_views:
            joined.append(x.float().cpu().numpy())
    del vec
    return (np.concatenate(probs), np.concatenate(joined)) if keep_views else np.concatenate(probs)


def best_threshold(y, p):
    """The threshold that maximises F1 on (y, p)."""
    P, R, T = precision_recall_curve(y, p)
    f1 = np.where(P + R > 0, 2 * P * R / np.maximum(P + R, 1e-12), 0)[:-1]
    return float(T[int(np.argmax(f1))])


def scores(y, p, th):
    """At the threshold: the confusion matrix, precision, recall and F1.
    From the probabilities: the loss (binary cross-entropy), ROC-AUC and AP."""
    f = p >= th
    pr, rc, f1, _ = precision_recall_fscore_support(y, f.astype(int), average="binary", zero_division=0)
    both = len(np.unique(y)) == 2                       # the two areas need both classes present
    return {"tp": int((f & (y == 1)).sum()), "fp": int((f & (y == 0)).sum()),
            "fn": int((~f & (y == 1)).sum()), "tn": int((~f & (y == 0)).sum()),
            "loss": float(log_loss(y, np.clip(p.astype(np.float64), 1e-7, 1 - 1e-7), labels=[0, 1])),
            "precision": float(pr), "recall": float(rc), "f1": float(f1),
            "roc_auc": float(roc_auc_score(y, p)) if both else float("nan"),
            "ap": float(average_precision_score(y, p)) if both else float("nan")}


print("model defined")


### Cell 6 — The text budget and a memory check
`CAP` is 64 windows per view per row, the budget run 2's pilot measured, and a micro-batch holds at
most 192 windows (3 × `CAP`, so that any one row fits). Gradient accumulation makes the micro-batch
size a matter of memory only: the optimiser step is the same however its 16 rows are split.

The cell runs one training micro-batch of the worst case, 192 windows of 512 tokens, and stops if the
GPU runs out of memory. It ends with how many rows have more windows than the budget, per view.

In [ ]:
POS_BY = {r: FIT[(LABEL[FIT] == 1) & (ROWS.repo.to_numpy()[FIT] == r)] for r in REPOS}
NEG_BY = {r: FIT[(LABEL[FIT] == 0) & (ROWS.repo.to_numpy()[FIT] == r)] for r in REPOS}
FIT_S = np.sort(np.random.default_rng(SEED).choice(FIT, min(FIT_SCORED, len(FIT)), replace=False))
print(pd.DataFrame({"refactored": {r: len(POS_BY[r]) for r in REPOS},
                    "not refactored": {r: len(NEG_BY[r]) for r in REPOS}}).to_string())


def row_windows(rows, views, cap):
    return sum(np.minimum(cap, WTOT[v][rows]) for v in views)


def micro_batches(rows, views, cap):
    """Rows, in the given order, grouped so that a micro-batch holds at most BATCH_WINDOWS windows."""
    limit, size = max(BATCH_WINDOWS, 3 * cap), row_windows(rows, views, cap)
    out, cur, used = [], [], 0
    for r, s in zip(rows, size):
        if cur and used + s > limit:
            out.append(cur)
            cur, used = [], 0
        cur.append(int(r))
        used += int(s)
    return out + [cur] if cur else out


def train_step(model, opt, parts, n, sched=None):
    """One optimiser step over n rows, given as micro-batches: their gradients are added up, then one step.
    Returns the mean loss of the n rows."""
    total = 0.0
    for batch in parts:
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            logit = model(batch)
        loss = nn.functional.binary_cross_entropy_with_logits(logit.float(), batch["y"].to(DEVICE), reduction="sum")
        (loss / n).backward()
        total += float(loss.detach())
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    opt.step()
    if sched is not None:
        sched.step()
    opt.zero_grad(set_to_none=True)
    return total / n


if DEVICE == "cuda":
    model = Forecaster(VIEWS).to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    n = BATCH_WINDOWS // len(VIEWS)
    worst = {"n": 1, "y": torch.zeros(1)}
    for v in VIEWS:
        ids, att = pad(v, [np.full(BODY[v], 100, dtype=np.int64)] * n)
        worst[v] = (ids, att, torch.zeros(n, dtype=torch.long), torch.zeros(1, dtype=torch.long))
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        train_step(model, opt, [worst], 1)
        print(f"\nmemory check: the worst micro-batch peaks at {torch.cuda.max_memory_allocated() / 1e9:.1f} GB "
              f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    except torch.cuda.OutOfMemoryError:
        raise SystemExit("out of memory on the worst micro-batch: this run needs a GPU like run 2's") from None
    finally:
        del model, opt
        gc.collect()
        torch.cuda.empty_cache()

print(f"\nCAP = {CAP} windows per view per row   BATCH_WINDOWS = {BATCH_WINDOWS}   ROWS_PER_STEP = {ROWS_PER_STEP}\n")
over = {v: {s: f"{(WTOT[v][rows] > CAP).mean():.1%}" for s, rows in
            (("train", FIT), ("validation", VAL), ("test", TEST))} for v in VIEWS}
print("rows with more windows than the budget (they read only the first CAP)")
print(pd.DataFrame(over).to_string())


### Cell 7 — One training run
`train_run(lr)` trains the model and returns its record. Per epoch:

1. a fresh draw, balanced per repository, taken 16 rows per optimiser step;
2. the validation rows are forecast, the F1-maximising threshold is found, and the five scores, the
   loss and the confusion matrix are computed;
3. the 20,000 sampled training rows are scored at that threshold;
4. each view's contribution on the validation rows is measured (the mean absolute Shapley value of
   cell 10, with the training sample as the average).

The best epoch by validation AP is kept in the checkpoints folder. Training stops after `PATIENCE` epochs without a
better validation AP, or after `EPOCHS`. After every epoch the full state is saved, so a disconnected
run resumes where it stopped. A finished run is written to `runs.jsonl` and is not trained again.

In [ ]:
RUNS = OUT / "runs.jsonl"


def finished():
    if not RUNS.exists():
        return {}
    return {r["key"]: r for r in (json.loads(l) for l in RUNS.read_text(encoding="utf-8").splitlines() if l.strip())}


def epoch_rows(ep):
    """For each repository, all its refactored training rows and as many of its non-refactored ones."""
    rng = np.random.default_rng(SEED + ep)
    take = []
    for r in REPOS:
        take += [POS_BY[r], rng.choice(NEG_BY[r], len(POS_BY[r]), replace=False)]
    return rng.permutation(np.concatenate(take))


def epoch_steps(ep):
    """The epoch's optimiser steps: groups of ROWS_PER_STEP rows, each split into micro-batches."""
    rows = epoch_rows(ep)
    return [micro_batches(rows[i:i + ROWS_PER_STEP], VIEWS, CAP) for i in range(0, len(rows), ROWS_PER_STEP)]


def view_phi(model, mean, X):
    """Each view's Shapley value per row (cell 10): its weights times the row's distance from the average."""
    w = model.out.weight.detach().float().cpu().numpy()[0]
    H = len(w) // len(VIEWS)
    return {v: (X[:, i * H:(i + 1) * H] - mean[i * H:(i + 1) * H]) @ w[i * H:(i + 1) * H]
            for i, v in enumerate(VIEWS)}


def save(obj, path):
    """Write to a temporary file, then rename, so a disconnect mid-write never leaves a broken checkpoint."""
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def train_run(lr):
    key = f"lr{lr:g}"
    if key in finished():
        print(f"{key}: finished earlier, skipped")
        return finished()[key]
    torch.manual_seed(SEED)
    model = Forecaster(VIEWS).to(DEVICE)
    enc = [p for n, p in model.named_parameters() if n.startswith("enc.")]
    rest = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    opt = torch.optim.AdamW([{"params": enc, "lr": lr}, {"params": rest, "lr": LR_HEAD}],
                            weight_decay=WEIGHT_DECAY)
    plan = [epoch_steps(ep) for ep in range(EPOCHS)]
    total = sum(map(len, plan))
    warm = max(1, int(WARMUP * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    last, best = CKPT / f"{key}_last.pt", CKPT / f"{key}_best.pt"
    start, best_score, bad, hist = 0, -1.0, 0, []
    if last.exists():
        st = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(st["model"])
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        start, best_score, bad, hist = st["epoch"] + 1, st["best_score"], st["bad"], st["hist"]
        print(f"{key}: resuming at epoch {start + 1}")

    for ep in range(start, EPOCHS):
        if bad >= PATIENCE:
            break
        model.train()
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0, loss = time.time(), []
        for i, step in enumerate(plan[ep]):
            n = sum(map(len, step))
            loss.append(train_step(model, opt, (make_batch(b, VIEWS, CAP) for b in step), n, sched))
            if i % 50 == 0:
                done = (time.time() - t0) / (i + 1)
                print(f"\r{key} epoch {ep + 1}: step {i:,}/{len(plan[ep]):,}  loss {np.mean(loss[-50:]):.3f}  "
                      f"{done * (len(plan[ep]) - i) / 60:.0f} min left", end="")
        peak = round(torch.cuda.max_memory_allocated() / 1e9, 1) if DEVICE == "cuda" else 0.0
        t1 = time.time()
        p_val, J_val = predict(model, VAL, CAP, keep_views=True)
        th = best_threshold(LABEL[VAL], p_val)
        val = scores(LABEL[VAL], p_val, th)
        t2 = time.time()
        p_fit, J_fit = predict(model, FIT_S, CAP, keep_views=True)
        fit = scores(LABEL[FIT_S], p_fit, th)
        phi = view_phi(model, J_fit.mean(0), J_val)
        del J_val, J_fit
        t3 = time.time()
        hist.append({"epoch": ep + 1, "optimiser steps": len(plan[ep]),
                     "train loss (balanced steps)": float(np.mean(loss)), "threshold": th,
                     **{f"train {k}": x for k, x in fit.items()}, **{f"validation {k}": x for k, x in val.items()},
                     **{f"validation |shap| {v}": float(np.abs(phi[v]).mean()) for v in VIEWS},
                     "training minutes": round((t1 - t0) / 60, 2), "validation minutes": round((t2 - t1) / 60, 2),
                     "train scoring minutes": round((t3 - t2) / 60, 2), "minutes": round((t3 - t0) / 60, 2),
                     "peak GPU GB": peak, "gpu": GPU})
        if val[SELECT] > best_score:
            best_score, bad = val[SELECT], 0
            save(model.state_dict(), best)
        else:
            bad += 1
        save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
              "epoch": ep, "best_score": best_score, "bad": bad, "hist": hist}, last)
        h = hist[-1]
        print(f"\r{key} epoch {ep + 1}: loss steps {h['train loss (balanced steps)']:.3f}  sample {fit['loss']:.3f}  "
              f"validation {val['loss']:.3f}   F1 train {fit['f1']:.3f} validation {val['f1']:.3f}   "
              f"AP train {fit['ap']:.3f} validation {val['ap']:.3f}   |shap| "
              + " ".join(f"{v} {h[f'validation |shap| {v}']:.2f}" for v in VIEWS)
              + f"   {'best' if bad == 0 else f'{bad} without improvement'}   {h['minutes']:.0f} min")

    top = max(hist, key=lambda h: h[f"validation {SELECT}"])
    rec = {"key": key, "views": VIEWS, "lr_encoder": lr, "cap": CAP, "rows_per_step": ROWS_PER_STEP,
           "select": SELECT, "best_epoch": top["epoch"], f"validation_{SELECT}": top[f"validation {SELECT}"],
           "threshold": top["threshold"], "weights": best.name, "history": hist}
    with open(RUNS, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    last.unlink(missing_ok=True)
    del model, opt
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rec


print("train_run defined")


### Cell 8 — The training run
One training run with the encoders' learning rate at 2e-5. Its best epoch by validation AP is the model.
`epochs.csv` holds every epoch.

In [ ]:
RUN = train_run(LR)
EPOCHS_DF = pd.DataFrame(RUN["history"])
EPOCHS_DF.to_csv(OUT / "epochs.csv", index=False)
print(f"\nbest epoch {RUN['best_epoch']}   validation {SELECT.upper()} {RUN[f'validation_{SELECT}']:.3f}\n")
cols = ["epoch", "train loss (balanced steps)", "train loss", "validation loss", "train f1", "validation f1",
        "train ap", "validation ap"] + [f"validation |shap| {v}" for v in VIEWS] + ["minutes", "peak GPU GB"]
print(EPOCHS_DF[cols].round(3).to_string(index=False))


### Cell 9 — The scores, the confusion matrices and the run times
The model forecasts the training, validation and test rows. The threshold is the one that maximises
F1 on the validation rows; it is applied unchanged to the training and test rows. This cell and cell 10
are the only ones that read the test rows' labels.

For each part and repository, `results.csv` holds the confusion matrix at the threshold (TP: refactored
and forecast; FP: forecast but not refactored; FN: refactored but not forecast; TN: neither), the loss,
precision, recall and F1, ROC-AUC and AP. Also writes `run_times.csv`, `forecasts.parquet` and
`run_config.json`.

In [ ]:
model = Forecaster(VIEWS).to(DEVICE)
model.load_state_dict(torch.load(CKPT / RUN["weights"], map_location=DEVICE))
PARTS_ = (("train", FIT), ("validation", VAL), ("test", TEST))
prob, took, JOINED = {}, {}, {}
for part, rows in PARTS_:
    t0 = time.time()
    prob[part], JOINED[part] = predict(model, rows, CAP, keep_views=True)
    took[part] = round((time.time() - t0) / 60, 2)
TH = best_threshold(LABEL[VAL], prob["validation"])
print(f"threshold {TH:.4f} (at training time {RUN['threshold']:.4f})")

results = []
for part, rows in PARTS_:
    repo = ROWS.repo.to_numpy()[rows]
    for name in REPOS + ["all"]:
        m = np.ones(len(rows), bool) if name == "all" else repo == name
        y = LABEL[rows][m]
        results.append({"part": part, "repo": name, "rows": int(m.sum()), "refactored": int(y.sum()),
                        **scores(y, prob[part][m], TH)})
RESULTS = pd.DataFrame(results)
RESULTS.to_csv(OUT / "results.csv", index=False)
pd.concat([ROWS.loc[rows, ["row_id", "repo", "release", "path", "label"]].assign(
    part=part, probability=prob[part], forecast=(prob[part] >= TH).astype(int)) for part, rows in PARTS_]
          ).to_parquet(OUT / "forecasts.parquet", index=False)

h = pd.DataFrame(RUN["history"])
TIMES = pd.concat([
    h[["epoch", "training minutes", "validation minutes", "train scoring minutes", "minutes", "peak GPU GB"]]
    .rename(columns={"validation minutes": "validation scoring minutes",
                     "train scoring minutes": "training-sample scoring minutes", "minutes": "epoch minutes"})
    .assign(stage="epoch"),
    pd.DataFrame([{"stage": f"final scoring, {p} rows", "epoch minutes": m} for p, m in took.items()]),
    pd.DataFrame([{"stage": "total", "training minutes": h["training minutes"].sum(),
                   "validation scoring minutes": h["validation minutes"].sum(),
                   "training-sample scoring minutes": h["train scoring minutes"].sum(),
                   "epoch minutes": h["minutes"].sum() + sum(took.values()),
                   "peak GPU GB": h["peak GPU GB"].max()}])], ignore_index=True)
TIMES = TIMES[["stage", "epoch", "training minutes", "validation scoring minutes", "training-sample scoring minutes",
               "epoch minutes", "peak GPU GB"]]
TIMES.to_csv(OUT / "run_times.csv", index=False)
json.dump({"dataset": "10.5281/zenodo.24117444", "encoders": ENCODER, "views": VIEWS,
           "lr_encoder": LR, "lr_head": LR_HEAD, "rows_per_step": ROWS_PER_STEP, "select": SELECT,
           "cap": CAP, "batch_windows": BATCH_WINDOWS, "threshold": TH,
           "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT, "warmup": WARMUP, "clip": CLIP,
           "epochs": EPOCHS, "patience": PATIENCE, "seed": SEED, "gpu": GPU,
           "run": {x: RUN[x] for x in ("key", "best_epoch", f"validation_{SELECT}")},
           "versions": {"python": platform.python_version(), "torch": torch.__version__,
                        "transformers": transformers.__version__, "scikit-learn": sklearn.__version__,
                        "pandas": pd.__version__}},
          open(OUT / "run_config.json", "w"), indent=1)

print("\n" + RESULTS.round(3).to_string(index=False))
print("\nCONFUSION MATRICES, all repositories (rows: actual; columns: forecast)")
for part, _ in PARTS_:
    r = RESULTS[(RESULTS.part == part) & (RESULTS.repo == "all")].iloc[0]
    print(f"\n{part}\n" + pd.DataFrame([[r.tp, r.fn], [r.fp, r.tn]], index=["refactored", "not refactored"],
                                       columns=["forecast refactored", "forecast not"]).to_string())
print("\nRUN TIME (minutes)\n" + TIMES.round(2).to_string(index=False))
print(f"\nwrote results.csv, run_times.csv, forecasts.parquet and run_config.json to {OUT}")


### Cell 10 — Each view's contribution: Shapley values
The classifier is one linear layer over the three view vectors joined, so the forecast's log-odds is a
sum of three parts, one per view. For a linear model the Shapley value of a group of inputs has an exact
closed form (Lundberg & Lee, 2017): the view's weights times how far the row's view vector lies from the
average one,

```latex
\phi_{v} = w_{v}^{\top}\,(x_{v} - \bar{x}_{v})
```

where the average is taken over the 20,000 sampled training rows. The three values and the base value
add up exactly to the row's log-odds; the cell stops if they do not. A positive value pushes that row's
forecast towards "refactored". No extra model is trained and no sampling is involved.

What this measures: how much each view moved this trained model's forecasts, row by row. What it does not
measure: how well a model trained without a view would do. That would need a model per combination of
views, which this experiment does not train.

Writes `view_shap_test.parquet` (one line per test row) and `view_shap.csv`: per repository and overall,
each view's mean absolute contribution, and its mean contribution for refactored and for other rows.

In [ ]:
w = model.out.weight.detach().float().cpu().numpy()[0]
b = float(model.out.bias.detach().float().cpu())
_, back = predict(model, FIT_S, CAP, keep_views=True)          # the average view vectors: training sample
mean = back.mean(0)
base = b + float(w @ mean)
X = JOINED["test"]
phi = view_phi(model, mean, X)
gap = np.abs(base + sum(phi.values()) - (X @ w + b)).max()
if gap > 1e-3:
    raise SystemExit(f"the Shapley values do not add up to the log-odds (largest gap {gap:.2e})")

S = ROWS.loc[TEST, ["row_id", "repo", "label"]].reset_index(drop=True).assign(
    probability=prob["test"], base=base, **{f"shap_{v}": phi[v] for v in VIEWS})
S.to_parquet(OUT / "view_shap_test.parquet", index=False)
summary = []
for name in REPOS + ["all"]:
    g = S if name == "all" else S[S.repo == name]
    for v in VIEWS:
        summary.append({"repo": name, "view": v, "mean |shap|": float(g[f"shap_{v}"].abs().mean()),
                        "mean shap, refactored": float(g[f"shap_{v}"][g.label == 1].mean()),
                        "mean shap, other": float(g[f"shap_{v}"][g.label == 0].mean())})
VIEW_SHAP = pd.DataFrame(summary)
VIEW_SHAP.to_csv(OUT / "view_shap.csv", index=False)
print(f"base value (log-odds) {base:.3f}   the values add up to every row's log-odds (largest gap {gap:.1e})\n")
print(VIEW_SHAP.pivot(index="repo", columns="view", values="mean |shap|").reindex(REPOS + ["all"])[VIEWS]
      .round(3).to_string())
print("\nmean contribution on the test rows, refactored vs other (log-odds)")
print(VIEW_SHAP[VIEW_SHAP.repo == "all"].set_index("view").iloc[:, 2:].round(3).to_string())


### Cell 11 — Curves and confusion matrices
Per epoch: the loss, F1, AP and each view's contribution. Training loss and F1 improving while the
validation ones stall or worsen is overfitting; the dotted line marks the epoch that was kept. Compare
the training-sample loss with the validation loss: both are measured on the natural share of
refactored rows. Then the confusion matrices of the three parts at the validation threshold.

In [ ]:
import matplotlib.pyplot as plt

h = pd.DataFrame(RUN["history"])
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
ax[0].plot(h.epoch, h["train loss (balanced steps)"], marker="o", label="training steps (balanced)")
ax[0].plot(h.epoch, h["train loss"], marker="o", label="training sample")
ax[0].plot(h.epoch, h["validation loss"], marker="o", label="validation")
ax[0].set_title("loss (binary cross-entropy)", fontsize=10)
for a, k in ((ax[1], "f1"), (ax[2], "ap")):
    a.plot(h.epoch, h[f"train {k}"], marker="o", label="training sample")
    a.plot(h.epoch, h[f"validation {k}"], marker="o", label="validation")
    a.set_title(k.upper() if k == "ap" else "F1", fontsize=10)
for v in VIEWS:
    ax[3].plot(h.epoch, h[f"validation |shap| {v}"], marker="o", label=v)
ax[3].set_title("each view's contribution, validation (mean |Shapley|)", fontsize=10)
for a in ax:
    a.axvline(RUN["best_epoch"], color="grey", linestyle=":")
    a.set_xlabel("epoch")
    a.set_xticks(h.epoch)
    a.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, (part, _) in zip(ax, PARTS_):
    r = RESULTS[(RESULTS.part == part) & (RESULTS.repo == "all")].iloc[0]
    M = np.array([[r.tp, r.fn], [r.fp, r.tn]])
    a.imshow(M / M.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            a.text(j, i, f"{M[i, j]:,}\n{M[i, j] / M[i].sum():.1%} of the row", ha="center", va="center",
                   color="white" if M[i, j] / M[i].sum() > 0.5 else "black", fontsize=9)
    a.set_xticks([0, 1], ["forecast refactored", "forecast not"])
    a.set_yticks([0, 1], ["refactored", "not refactored"])
    a.set_title(f"{part}: {int(r.rows):,} rows, threshold {TH:.3f}", fontsize=10)
fig.tight_layout()
fig.savefig(OUT / "confusion.png", dpi=150)
plt.show()
print(f"wrote curves.png and confusion.png to {OUT}")
